# Ace Combat AWACS Simulation
**Parallelism via Apache Spark, CUDA, and Machine Learning**

This notebook executes:
- **PySpark** for big data processing and Random Forest ML threat scoring
- **CUDA (C++)** for parallel missile trajectory math
- **Interactive 3D Visualization** via Plotly

Project: **Schryzon/mpyCUDA**  
Course: **Parallel Processing A**



> ### 🛑 GPU REQUIRED!
> **This project REQUIRES a CUDA-capable GPU (NVIDIA T4 or better).**
> Go to **Runtime** → **Change runtime type**, select **T4 GPU**, and click **Save**.



## 1. Environment Setup & Data Generation


In [1]:
# Mount Google Drive and Setup Repository
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Jay-IF24-mpyCUDA'
REPO_URL   = 'https://github.com/Schryzon/mpyCUDA.git'

if not os.path.exists(DRIVE_PATH):
    !git clone "{REPO_URL}" "{DRIVE_PATH}"
else:
    !git -C "{DRIVE_PATH}" pull

WORK_DIR = '/content/mpyCUDA'
if not os.path.exists(WORK_DIR):
    !ln -s "{DRIVE_PATH}" "{WORK_DIR}"

%cd {WORK_DIR}/Ace-Combat-AWACS-Simulation



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/drive/MyDrive/Jay-IF24-mpyCUDA/Ace-Combat-AWACS-Simulation


In [2]:
!apt-get update -qq
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

!pip install pyspark plotly -q

!java -version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [3]:
# Generate Synthetic Radar Data (1,000,000 bogeys)
!python scripts/data_gen.py 1000000

Generating 1000000 synthetic radar records...
Data saved to radar_data.csv successfully.


## 2. Spark MLlib - Threat Prioritization


In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Start cleanly without calling SparkContext.getOrCreate() first
spark = SparkSession.builder \
    .appName("AWACS_Threat_Scoring") \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.local.dir", "/tmp/spark-temp") \
    .getOrCreate()

print("Spark Session created successfully.")


Spark Session created successfully.


In [5]:
# Load Data
df = spark.read.csv("radar_data.csv", header=True, inferSchema=True)

from pyspark.sql.functions import col, sqrt, atan2, degrees, abs as pyspark_abs, when

# Feature 1: Distance to base
df = df.withColumn("distance", sqrt(col("x")**2 + col("y")**2))

# Feature 2: Heading Difference (Are they pointing at us?)
# Calculate angle_to_base = atan2(-y, -x) and convert to degrees 0-360
df = df.withColumn("angle_to_base", (degrees(atan2(-col("y"), -col("x"))) + 360) % 360)
df = df.withColumn("raw_diff", pyspark_abs(col("heading") - col("angle_to_base")))
df = df.withColumn("heading_diff", when(col("raw_diff") > 180, 360 - col("raw_diff")).otherwise(col("raw_diff")))

# Assemble features using our new, calculated math!
assembler = VectorAssembler(
    inputCols=["distance", "altitude", "velocity", "heading_diff"],
    outputCol="features"
)
data = assembler.transform(df)



In [6]:
# Train Random Forest Model
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(labelCol="threat_label", featuresCol="features", numTrees=20)
print("Training Random Forest Model (Distributed)...")
model = rf.fit(train_data)

# Evaluate
predictions = model.transform(test_data)
evaluator = MulticlassClassificationEvaluator(labelCol="threat_label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")



Training Random Forest Model (Distributed)...
Model Accuracy: 82.59%


In [7]:
# Filter the most critical threats (Prediction == 3)
critical_df = predictions.filter(col("prediction") == 3.0)
top_threats = critical_df.orderBy("distance").limit(1000).toPandas()

print(f"Found {len(top_threats)} critical targets.")
top_threats.head()



Found 1000 critical targets.


,bogey_id,x,y,altitude,velocity,heading,threat_label,distance,angle_to_base,raw_diff,heading_diff,features,rawPrediction,probability,prediction
0,85906,-8489.15,-5363.28,8832.00,719.97,34.30,3,10041.436156,32.283901,2.016099,2.016099,"[10041.436156292584, 8832.0, 719.97, 2.0160985...","[0.25353335852072917, 1.2047013630440233, 9.26...","[0.012676667926036458, 0.060235068152201164, 0...",3.0
1,477577,6290.39,7827.18,12399.77,785.07,230.62,3,10041.601123,231.212559,0.592559,0.592559,"[10041.601122555108, 12399.77, 785.07, 0.59255...","[0.2523878914433688, 0.9767937603358925, 8.425...","[0.01261939457216844, 0.04883968801679463, 0.4...",3.0
2,56812,-8622.69,5168.08,2104.28,744.10,327.84,3,10052.852019,329.063289,1.223289,1.223289,"[10052.85201932765, 2104.28, 744.1, 1.22328946...","[0.2538795142439942, 0.8255434974916399, 8.442...","[0.012693975712199709, 0.041277174874582, 0.42...",3.0
3,777029,-2897.37,-9635.72,1692.50,775.77,75.91,3,10061.901055,73.264464,2.645536,2.645536,"[10061.901054736127, 1692.5, 775.77, 2.6455363...","[0.2538795142439942, 0.8255434974916399, 8.442...","[0.012693975712199709, 0.041277174874582, 0.42...",3.0
4,194269,10072.36,-73.09,9867.26,845.79,179.86,3,10072.625185,179.584241,0.275759,0.275759,"[10072.625185010113, 9867.26, 845.79, 0.275759...","[0.2523878914433688, 0.9767937603358925, 8.425...","[0.01261939457216844, 0.04883968801679463, 0.4...",3.0


## 3. CUDA - Parallel Interception Trajectories


In [8]:
# Compile the CUDA C++ kernel into a shared library
!nvcc -Xcompiler -fPIC -shared -o libtrajectory.so scripts/trajectory_math.cu
print("CUDA Kernel compiled to libtrajectory.so")



nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CUDA Kernel compiled to libtrajectory.so


In [9]:
# Execute CUDA via ctypes
import ctypes
import numpy as np

# Load the shared library
lib = ctypes.CDLL('./libtrajectory.so')

# Define argument types
lib.calculate_interception.argtypes = [
    ctypes.c_int,
    ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_int) # Evasions counter
]

num_targets = len(top_threats)
x_arr = np.array(top_threats['x'], dtype=np.float32)
y_arr = np.array(top_threats['y'], dtype=np.float32)
z_arr = np.array(top_threats['altitude'], dtype=np.float32)
v_arr = np.array(top_threats['velocity'], dtype=np.float32)
h_arr = np.array(top_threats['heading'], dtype=np.float32)

tti = np.zeros(num_targets, dtype=np.float32)
int_x = np.zeros(num_targets, dtype=np.float32)
int_y = np.zeros(num_targets, dtype=np.float32)
int_z = np.zeros(num_targets, dtype=np.float32)
evasions = ctypes.c_int(0)

print(f"Sending {num_targets} targets to CUDA GPU...")
lib.calculate_interception(
    num_targets,
    x_arr.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    y_arr.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    z_arr.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    v_arr.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    h_arr.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    tti.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    int_x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    int_y.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    int_z.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
    ctypes.byref(evasions)
)

top_threats['tti'] = tti
top_threats['int_x'] = int_x
top_threats['int_y'] = int_y
top_threats['int_z'] = int_z

print("CUDA computation complete!")
print(f"CRITICAL ALERT: {evasions.value} targets successfully evaded our SAM network!")
top_threats[['bogey_id', 'distance', 'tti']].head()



Sending 1000 targets to CUDA GPU...
CUDA computation complete!
CRITICAL ALERT: 0 targets successfully evaded our SAM network!


,bogey_id,distance,tti
0,85906,10041.436156,9.911970
1,477577,10041.601123,12.569629
2,56812,10052.852019,36.387886
3,777029,10061.901055,8.990533
4,194269,10072.625185,26.721062


## 4. AWACS Callouts & Visualization


In [10]:
# Generate AWACS Callouts for the top 5 threats
import math

def get_clock_position(x, y):
    # Allied base is (0,0), looking North (+y)
    # Target position is (x, y)
    angle_rad = math.atan2(x, y) # Angle from North, clockwise
    angle_deg = (math.degrees(angle_rad) + 360) % 360
    clock = int(round(angle_deg / 30.0))
    if clock == 0:
        clock = 12
    return clock

def get_elevation(z):
    if z > 10000: return "high"
    elif z < 3000: return "low"
    else: return "level"

print("\n===== AWACS ALERTS =====")
for i, row in top_threats.head(5).iterrows():
    clock = get_clock_position(row['x'], row['y'])
    elevation = get_elevation(row['altitude'])
    print(f"AWACS: \"Bogey, {clock} o'clock, {elevation}! Target ID {int(row['bogey_id'])}, distance {row['distance']/1000:.1f} km.\"")
print("========================\n")




===== AWACS ALERTS =====
AWACS: "Bogey, 8 o'clock, level! Target ID 85906, distance 10.0 km."
AWACS: "Bogey, 1 o'clock, high! Target ID 477577, distance 10.0 km."
AWACS: "Bogey, 10 o'clock, low! Target ID 56812, distance 10.1 km."
AWACS: "Bogey, 7 o'clock, low! Target ID 777029, distance 10.1 km."
AWACS: "Bogey, 3 o'clock, level! Target ID 194269, distance 10.1 km."



In [11]:
# Interactive 3D Visualization with Plotly
import plotly.graph_objects as go
import numpy as np

# Plot the Base
fig = go.Figure(data=[go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=10, color='green', symbol='diamond'),
    name='Allied Base'
)])

# Generate Radar Dome (Sphere)
theta = np.linspace(0, 2.*np.pi, 50)
phi = np.linspace(0, np.pi/2, 50) # Hemisphere (only above ground)
theta, phi = np.meshgrid(theta, phi)
r = 100000 # 100km Radar Range

x_dome = r * np.sin(phi) * np.cos(theta)
y_dome = r * np.sin(phi) * np.sin(theta)
z_dome = r * np.cos(phi)

fig.add_trace(go.Surface(
    x=x_dome, y=y_dome, z=z_dome,
    opacity=0.1,
    colorscale=[[0, 'rgba(0,255,0,0.1)'], [1, 'rgba(0,255,0,0.1)']],
    showscale=False,
    name='Radar Dome (100km)'
))

# Plot Critical Bogeys
fig.add_trace(go.Scatter3d(
    x=top_threats['x'], y=top_threats['y'], z=top_threats['altitude'],
    mode='markers',
    marker=dict(size=3, color='red'),
    name='Critical Bogeys'
))

# Plot the top 5 Interception Paths
for i, row in top_threats.head(5).iterrows():
    if row['tti'] > 0:
        fig.add_trace(go.Scatter3d(
            x=[0, row['int_x']], y=[0, row['int_y']], z=[0, row['int_z']],
            mode='lines',
            line=dict(color='yellow', width=2, dash='dash'),
            name=f'Missile Trajectory {int(row["bogey_id"])}'
        ))

fig.update_layout(
    title='AWACS Radar Space & SAM Interception Trajectories',
    scene=dict(
        xaxis_title='X (meters)',
        yaxis_title='Y (meters)',
        zaxis_title='Altitude (meters)',
        aspectmode='data'
    ),
    template='plotly_dark'
)

fig.show()

